# Day 18 — Disaggregation: Prefill/Decode Split
> *100 Days of Inference* | Layer: **Runtime** | Book: *Inference Engineering* Ch 5.5 (pp. 148–151)

**Prerequisite:** Day 17 (Model Parallelism)

## What problem does this solve?

Even with disaggregation covered in Day 12 (Dynamo), production engineers need to understand the full mechanics: how the prefill engine produces KV cache, transfers it via NIXL, and how decode workers consume it. The interaction between disaggregation, KV cache size, and network bandwidth determines whether this technique helps or hurts.

## Concept Overview

Disaggregation separates inference into two specialized roles:

**Prefill engine:** Takes input sequence, computes KV cache, generates first token. Compute-bound.
**Decode engine:** Takes KV cache from prefill, generates subsequent tokens. Memory-bound.

The three-step pipeline:
1. Prefill engine processes input → KV cache + first token
2. KV cache transferred to decode engine (NIXL over NVLink/InfiniBand)
3. Decode engine generates remaining tokens

**Conditional disaggregation:** The decode engine first checks if the request can be handled locally (cache hit or short input). Only sends to prefill engine if truly necessary.

**xPyD notation:** 5P3D = 5 prefill workers + 3 decode workers for one model deployment.

In [1]:
!pip install -q numpy matplotlib torch 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10
GPU Memory: 128.5 GB


## Part 1: Prefill vs Decode Resource Profiles

Prefill and decode have fundamentally different hardware needs:

- **Prefill:** Compute-bound (many tokens in parallel → high arithmetic intensity). Wants more Tensor Cores.
- **Decode:** Memory-bound (one token at a time → low arithmetic intensity). Wants more memory bandwidth.

When both run on the same GPU, they compete. Disaggregation assigns each phase to specialized hardware.

In [2]:
# Compute the optimal TP degree for prefill vs decode workers
# Prefill: higher TP reduces TTFT (compute-bound, scales well with TP)
# Decode: lower TP is fine (memory-bound, communication overhead hurts)

def model_latency(n_tokens, hidden, layers, n_gpus_tp,
                  flops_per_gpu_tflops=989, bw_per_gpu_gbs=3350,
                  bytes_per_param=2, comm_overhead_fraction=0.05):
    """Estimate prefill and decode latency for given TP degree."""
    params_per_layer = 4 * hidden * hidden  # simplified (attn + FFN)
    total_params = params_per_layer * layers

    # Prefill: compute-bound, scales with 1/n_gpus (more GPUs = more compute)
    flops_prefill = 2 * total_params * n_tokens  # ~2 * params per token
    compute_seconds = flops_prefill / (flops_per_gpu_tflops * 1e12 * n_gpus_tp)
    comm_overhead = comm_overhead_fraction * (n_gpus_tp - 1)
    ttft_ms = (compute_seconds + compute_seconds * comm_overhead) * 1000

    # Decode: memory-bound, scales with bandwidth
    weights_bytes = total_params * bytes_per_param / n_gpus_tp
    decode_seconds = weights_bytes / (bw_per_gpu_gbs * 1e9)
    decode_ms = (decode_seconds + decode_seconds * comm_overhead) * 1000

    return ttft_ms, decode_ms

tp_degrees = [1, 2, 4, 8]
input_len = 2000
hidden = 4096
layers = 32

print("Prefill vs Decode latency by TP degree (7B-class model)")
print(f"Input: {input_len} tokens, hidden={hidden}, layers={layers}")
print()
print(f"{'TP degree':>10} {'TTFT (ms)':>12} {'Decode step (ms)':>18} {'Tradeoff'}")
print("-" * 60)

for tp in tp_degrees:
    ttft, decode = model_latency(input_len, hidden, layers, tp)
    note = "Optimal for latency" if tp == 4 else ""
    print(f"{tp:>10} {ttft:>12.1f} {decode:>18.2f}  {note}")

print()
print("Key insight: for disaggregated serving, prefill workers should use higher TP")
print("than decode workers. Prefill scales well; decode communication overhead hurts more.")

Prefill vs Decode latency by TP degree (7B-class model)
Input: 2000 tokens, hidden=4096, layers=32

 TP degree    TTFT (ms)   Decode step (ms) Tradeoff
------------------------------------------------------------
         1          8.7               1.28  
         2          4.6               0.67  
         4          2.5               0.37  Optimal for latency
         8          1.5               0.22  

Key insight: for disaggregated serving, prefill workers should use higher TP
than decode workers. Prefill scales well; decode communication overhead hurts more.


## Part 2: When to Use Disaggregation

The book gives clear guidance on when disaggregation is worth the engineering complexity:

1. Serving large traffic (100M-1B tokens/day)
2. Model has 100B+ parameters
3. Traffic is prefill-heavy (long input sequences)

If any of these aren't true, the overhead of disaggregation (KV transfer, extra infrastructure, more complex routing) isn't worth the benefit.

In [3]:
import matplotlib.pyplot as plt
import numpy as np

# Model: when does disaggregation pay off?
# Simple model: disaggregation reduces TTFT proportionally to prefill fraction

def disaggregation_benefit(
    daily_tokens, model_params_b, avg_input_len, avg_output_len,
    disagg_overhead_ms=5,  # KV transfer overhead
):
    """
    Estimate TTFT improvement from disaggregation.
    Returns: (pct_improvement, is_worth_it)
    """
    prefill_fraction = avg_input_len / (avg_input_len + avg_output_len)

    # TTFT improvement: disagg removes decode competition during prefill
    # Benefit scales with prefill fraction and traffic
    ttft_improvement_pct = prefill_fraction * 40  # ~40% improvement at high prefill fraction

    # When disaggregation pays off
    sufficient_traffic = daily_tokens > 100e6
    large_model = model_params_b > 100
    heavy_prefill = prefill_fraction > 0.7

    score = sum([sufficient_traffic, large_model, heavy_prefill])
    is_worth_it = score >= 2

    return ttft_improvement_pct, is_worth_it, {
        'traffic': sufficient_traffic,
        'model_size': large_model,
        'prefill_heavy': heavy_prefill,
        'score': score
    }

use_cases = [
    ("Code editor (1T tokens/day, 671B, long input)",  1e12, 671, 2000, 100),
    ("Chatbot (500M tokens/day, 70B, balanced)",       500e6, 70, 200, 200),
    ("Small API (10M tokens/day, 7B, short input)",    10e6, 7, 50, 150),
    ("RAG system (200M tokens/day, 70B, long input)", 200e6, 70, 1000, 200),
    ("Translation (50M tokens/day, 7B, balanced)",    50e6, 7, 100, 100),
]

print("Should you use disaggregation?")
print()
print(f"{'Use case':<45} {'TTFT gain':>10} {'Traffic':>8} {'Model':>6} {'Prefill':>8} {'Worth it?':>10}")
print("-" * 95)
for name, daily, params_b, in_len, out_len in use_cases:
    benefit, worth_it, flags = disaggregation_benefit(daily, params_b, in_len, out_len)
    t = '✓' if flags['traffic'] else '✗'
    m = '✓' if flags['model_size'] else '✗'
    p = '✓' if flags['prefill_heavy'] else '✗'
    print(f"{name[:45]:<45} {benefit:>9.0f}% {t:>8} {m:>6} {p:>8} {'YES' if worth_it else 'NO':>10}")

Should you use disaggregation?

Use case                                       TTFT gain  Traffic  Model  Prefill  Worth it?
-----------------------------------------------------------------------------------------------
Code editor (1T tokens/day, 671B, long input)        38%        ✓      ✓        ✓        YES
Chatbot (500M tokens/day, 70B, balanced)             20%        ✓      ✗        ✗         NO
Small API (10M tokens/day, 7B, short input)          10%        ✗      ✗        ✗         NO
RAG system (200M tokens/day, 70B, long input)        33%        ✓      ✗        ✓        YES
Translation (50M tokens/day, 7B, balanced)           20%        ✗      ✗        ✗         NO


## Try These Experiments

1. **KV cache transfer time vs benefit:** For a 70B model at FP16, calculate the KV cache transfer time over NVLink (900 GB/s) for input lengths of [512, 2048, 8192, 32768]. Compare this to the prefill time savings. At what input length does disaggregation become worthwhile?

2. **Optimal xPyD for mixed traffic:** You have 16 GPUs and traffic that is 70% short queries (input=100, output=200) and 30% long queries (input=5000, output=100). Find the xPyD ratio that maximizes P95 TTFT across the entire traffic mix.

3. **Queue depth management:** Disaggregation introduces a prefill queue. Model the queue depth as a function of request arrival rate and prefill throughput. At what arrival rate does the queue cause TTFT to exceed 2x the uncongested TTFT?

## Key Takeaways

- Disaggregation is powerful but only justified at scale (>100M tokens/day, >100B params, prefill-heavy traffic).
- Prefill workers should use higher TP (compute matters more); decode workers use lower TP (memory bandwidth matters more).
- NIXL makes KV transfer overhead negligible on NVLink — disaggregation is nearly free within a node.
- Conditional disaggregation avoids overhead for requests that can be handled locally.
- **What's next:** Day 19 — GPU Architecture: SMs, memory hierarchy, HBM — understanding the hardware beneath all these optimizations.

## References
- *Inference Engineering* Ch 5.5 (pp. 148–151)